<a href="https://colab.research.google.com/github/RajBhatta67/measles-rubella-project/blob/main/Measles_and_Rubles_Research_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Checkpoint #1


In [7]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets from GitHub raw URLs
yearly_df = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_yearly.csv')
monthly_df = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_monthly.csv')


In [8]:
# Rename verbose columns to something workable
yearly_df= yearly_df.rename(columns = {'measles_incidence_rate_per_1000000_total_population': 'measles_incidence_rate',
    'rubella_incidence_rate_per_1000000_total_population': 'rubella_incidence_rate',
    'annualized_population_most_recent_year_only': 'annualized_population',
    'discarded_non_measles_rubella_cases_per_100000_total_population': 'discarded_rate',
    'total_suspected_measles_rubella_cases': 'total_suspected',
    'discarded_cases': 'discarded_total', })

In [9]:
yearly_number_columns = [
    'total_population',
    'annualized_population',
    'total_suspected',
    'measles_total',
    'measles_lab_confirmed',
    'measles_epi_linked',
    'measles_clinical',
    'measles_incidence_rate',
    'rubella_total',
    'rubella_lab_confirmed',
    'rubella_epi_linked',
    'rubella_clinical',
    'rubella_incidence_rate',
    'discarded_total',
    'discarded_rate',
]

# go through each one and force it to be a number
# if a value can't be converted it becomes NaN (blank)
for col in yearly_number_columns:
    if col in yearly_df.columns:
        yearly_df[col] = pd.to_numeric(yearly_df[col], errors='coerce')
# same thing for the monthly file
monthly_number_columns = [
    'measles_suspect',
    'measles_clinical',
    'measles_epi_linked',
    'measles_lab_confirmed',
    'measles_total',
    'rubella_clinical',
    'rubella_epi_linked',
    'rubella_lab_confirmed',
    'rubella_total',
    'discarded',
]

for col in monthly_number_columns:
    if col in monthly_df.columns:
        monthly_df[col] = pd.to_numeric(monthly_df[col], errors='coerce')

In [10]:
# year and month must be whole numbers
yearly_df['year']  = yearly_df['year'].astype(int)
monthly_df['year'] = monthly_df['year'].astype(int)
monthly_df['month'] = monthly_df['month'].astype(int)

In [11]:
# Check for missing values
yearly_df.isna().sum()

,0
region,0
country,0
iso3,0
year,0
total_population,0
annualized_population,0
total_suspected,87
measles_total,0
measles_lab_confirmed,0
measles_epi_linked,0


In [12]:
#Amount of data missing
missing_pct = yearly_df.isna().mean() * 100
print(missing_pct.round(1))

region                     0.0
country                    0.0
iso3                       0.0
year                       0.0
total_population           0.0
annualized_population      0.0
total_suspected            3.7
measles_total              0.0
measles_lab_confirmed      0.0
measles_epi_linked         0.0
measles_clinical           0.0
measles_incidence_rate     0.0
rubella_total              0.0
rubella_lab_confirmed      0.0
rubella_epi_linked         0.0
rubella_clinical           0.0
rubella_incidence_rate     0.0
discarded_total            3.7
discarded_rate            17.1
dtype: float64


In [ ]:
most_recent_year = yearly_df['year'].max()
yearly_df['is_partial_year'] = yearly_df['year'] >= most_recent_year
yearly_df['is_covid_disrupted'] = yearly_df['year'].isin([2020, 2021])
yearly_df['in_core_window'] = (
    (yearly_df['year'] >= 2012) &
    (yearly_df['year'] <= 2019) &
    (yearly_df['is_partial_year'] == False)
)

# where the measles surge is happening
yearly_df['in_surge_window'] = (
    (yearly_df['year'] >= 2022) &
    (yearly_df['year'] <= 2024)
)

In [ ]:

# I will use this to color-code charts by time period

yearly_df['period'] = 'Pre-COVID (2012-2019)'  # default
yearly_df.loc[yearly_df['year'].isin([2020, 2021]), 'period'] = 'COVID (2020-2021)'
yearly_df.loc[yearly_df['year'] >= 2022, 'period'] = 'Post-COVID (2022-2024)'

In [ ]:
MIN_CASES = 20

def safe_divide(top, bottom):
    # divides two columns but returns NaN instead of crashing
    # when denominator is 0 or missing
    result = []
    for t, b in zip(top, bottom):
        if pd.isna(t) or pd.isna(b) or b <= 0:
            result.append(np.nan)
        else:
            result.append(t / b)
    return result


# measles
yearly_df['measles_lab_share']      = safe_divide(yearly_df['measles_lab_confirmed'], yearly_df['measles_total'])
yearly_df['measles_clinical_share'] = safe_divide(yearly_df['measles_clinical'],      yearly_df['measles_total'])

# rubella
yearly_df['rubella_lab_share']      = safe_divide(yearly_df['rubella_lab_confirmed'], yearly_df['rubella_total'])
yearly_df['rubella_clinical_share'] = safe_divide(yearly_df['rubella_clinical'],      yearly_df['rubella_total'])

# log incidence — fixes the skew from a few countries having massive outbreaks
yearly_df['measles_log_incidence'] = np.log1p(yearly_df['measles_incidence_rate'])
yearly_df['rubella_log_incidence']  = np.log1p(yearly_df['rubella_incidence_rate'])

# flag rows where we have enough cases to trust the ratio
yearly_df['measles_denom_ok'] = yearly_df['measles_total'] >= MIN_CASES
yearly_df['rubella_denom_ok']  = yearly_df['rubella_total']  >= MIN_CASES

In [ ]:
yearly_df = yearly_df.sort_values(['country', 'year']).reset_index(drop=True)

# each country's own typical measles incidence
yearly_df['country_median_incidence'] = (
    yearly_df
    .groupby('country')['measles_incidence_rate']
    .transform('median')
)

# outbreak = more than 2x their own normal AND at least 20 cases
yearly_df['measles_outbreak_year'] = (
    (yearly_df['measles_incidence_rate'] > 2.0 * yearly_df['country_median_incidence']) &
    (yearly_df['measles_total'] >= MIN_CASES)
)

print("\noutbreak years flagged:", yearly_df['measles_outbreak_year'].sum())

In [ ]:
# Build the measles rows
measles_df = yearly_df[['country', 'iso3', 'region', 'year', 'period', 'total_population']].copy()
measles_df['disease'] = 'measles'
measles_df['total_cases'] = yearly_df['measles_total']
measles_df['lab_confirmed'] = yearly_df['measles_lab_confirmed']
measles_df['epi_linked'] = yearly_df['measles_epi_linked']
measles_df['clinical'] = yearly_df['measles_clinical']
measles_df['incidence_rate'] = yearly_df['measles_incidence_rate']
measles_df['log_incidence']  = yearly_df['measles_log_incidence']
measles_df['lab_share']      = yearly_df['measles_lab_share']
measles_df['clinical_share'] = yearly_df['measles_clinical_share']
measles_df['denom_ok']       = yearly_df['measles_denom_ok']

# Build the rubella rows
rubella_df = yearly_df[['country', 'iso3', 'region', 'year', 'period', 'total_population']].copy()
rubella_df['disease'] = 'rubella'
rubella_df['total_cases'] = yearly_df['rubella_total']
rubella_df['lab_confirmed'] = yearly_df['rubella_lab_confirmed']
rubella_df['epi_linked'] = yearly_df['rubella_epi_linked']
rubella_df['clinical'] = yearly_df['rubella_clinical']
rubella_df['incidence_rate'] = yearly_df['rubella_incidence_rate']
rubella_df['log_incidence']  = yearly_df['rubella_log_incidence']
rubella_df['lab_share']      = yearly_df['rubella_lab_share']
rubella_df['clinical_share'] = yearly_df['rubella_clinical_share']
rubella_df['denom_ok']       = yearly_df['rubella_denom_ok']

# Stack them into one long DataFrame
tidy_df = pd.concat([measles_df, rubella_df], ignore_index = True)


In [ ]:
monthly_df['measles_reported'] = monthly_df['measles_total'].notna()
monthly_df['rubella_reported']  = monthly_df['rubella_total'].notna()

completeness = monthly_df.groupby(['country', 'year']).agg(
    measles_months_reported = ('measles_reported', 'sum'),
    rubella_months_reported = ('rubella_reported', 'sum'),
).reset_index()

completeness['measles_completeness'] = completeness['measles_months_reported'] / 12
completeness['rubella_completeness']  = completeness['rubella_months_reported'] / 12

The research question im leaning towards:
To what extent is the reported divergence between measles and rubella incidence across 193 countries (2012–2024) attributable to rubella incidence being more dependent on a country’s lab-confirmation share than measles incidence is?


Checkpoint #2
